In [2]:
import pandas as pd

prices = pd.Series([100, 102, 99, 101, 104], index=pd.date_range("2024-01-01", periods=5))
signal = pd.Series([1, 1, 0, 1, 1], index=prices.index)

market_return = prices.pct_change().fillna(0)
position = signal.shift(1).fillna(0).astype(int)
strategy_return = market_return * position

pd.DataFrame({
    "price": prices,
    "signal": signal,
    "position": position,
    "market": market_return.round(4),
    "strategy": strategy_return.round(4),
})

,price,signal,position,market,strategy
2024-01-01,100,1,0,0.0000,0.0000
2024-01-02,102,1,1,0.0200,0.0200
2024-01-03,99,0,1,-0.0294,-0.0294
2024-01-04,101,1,0,0.0202,0.0000
2024-01-05,104,1,1,0.0297,0.0297


In [6]:
from backtester.data import load_prices
from backtester.strategy import sma_crossover

spy = load_prices('SPY', '2015-01-01', '2026-01-01')
signals = sma_crossover(spy['Adj Close'])

market_returns = spy['Adj Close'].pct_change().fillna(0)
positions = signals.shift(1).fillna(0).astype(int)
strategy_returns = market_returns * positions

results = pd.DataFrame({
    "price": spy['Adj Close'],
    "signal": signals,
    "position": positions,
    "market": market_returns.round(4),
    "strategy": strategy_returns.round(4),
})

results.loc['2020-02-28':'2020-03-06'].round(4)

Loaded SPY from cache: SPY_2015-01-01_2026-01-01.csv


,price,signal,position,market,strategy
Date,,,,,
2020-02-28,269.3779,1,1,-0.0042,-0.0042
2020-03-02,281.0436,1,1,0.0433,0.0433
2020-03-03,272.9968,0,1,-0.0286,-0.0286
2020-03-04,284.4715,0,0,0.0420,0.0000
2020-03-05,275.0152,0,0,-0.0332,-0.0000
2020-03-06,270.4689,0,0,-0.0165,-0.0000


In [8]:
cost_rate = 0.001

trades = positions.diff().abs().fillna(0)
cost = trades * cost_rate
net_return = strategy_returns - cost

pd.DataFrame({
    "position": positions,
    "trades": trades,
    "cost": cost.round(4),
    "strategy": strategy_returns.round(4),
    "net": net_return.round(4),
})

,position,trades,cost,strategy,net
Date,,,,,
2015-01-02,0,0.0,0.0,0.0000,0.0000
2015-01-05,0,0.0,0.0,-0.0000,-0.0000
2015-01-06,0,0.0,0.0,-0.0000,-0.0000
2015-01-07,0,0.0,0.0,0.0000,0.0000
2015-01-08,0,0.0,0.0,0.0000,0.0000
...,...,...,...,...,...
2025-12-24,1,0.0,0.0,0.0035,0.0035
2025-12-26,1,0.0,0.0,-0.0001,-0.0001
2025-12-29,1,0.0,0.0,-0.0036,-0.0036


In [9]:
results["trades"] = results["position"].diff().abs().fillna(0)
results["cost"] = results["trades"] * cost_rate

In [10]:
results["trades"].sum()

np.float64(51.0)

In [11]:
results["cost"].sum()

np.float64(0.051000000000000004)

In [12]:
results.loc["2016-05-25":"2016-06-01"].round(4)

,price,signal,position,market,strategy,trades,cost
Date,,,,,,,
2016-05-25,176.9059,1,1,0.0068,0.0068,0.0,0.000
2016-05-26,176.9566,0,1,0.0003,0.0003,0.0,0.000
2016-05-27,177.7175,1,0,0.0043,0.0000,1.0,0.001
2016-05-31,177.3793,1,1,-0.0019,-0.0019,1.0,0.001
2016-06-01,177.7427,1,1,0.0020,0.0020,0.0,0.000
